In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm
 
 
insurance_df= pd.read_csv(r'C:\Users\migle\Desktop\Python\insurance.csv')
 
insurance_df.head()
 

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [5]:
# Feature Engineering
 
insurance_df = (
    pd.get_dummies(insurance_df, drop_first=True)
    .assign(
        age_sq=lambda x: x["age"] ** 2,
        smoker_bmi_int=lambda x: x["smoker_yes"] * x["bmi"]
    )
)
 

In [6]:
# Data Splitting
 
from sklearn.model_selection import train_test_split
 
X = sm.add_constant(insurance_df.drop("charges", axis=1))
y = insurance_df["charges"]
 
# 1) Train+Valid ir Test (80/20)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2023
)
 
# 2) Train ir Valid iš Train+Valid (75/25 iš 80% => 60/20 galutiniame)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=2023
)

In [8]:
# Standardization
 
from sklearn.preprocessing import StandardScaler
 
scaler = StandardScaler()
 
cols = X_train.columns.drop("const")
 
X_tr_scaled = X_train.copy()
X_val_scaled = X_valid.copy()
X_te_scaled = X_test.copy()
 
X_tr_scaled[cols] = scaler.fit_transform(X_train[cols])
X_val_scaled[cols] = scaler.transform(X_valid[cols])
X_te_scaled[cols] = scaler.transform(X_test[cols])

In [9]:
X_train.head()

,const,age,bmi,children,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest,age_sq,smoker_bmi_int
680,1.0,21,17.400,1,False,False,False,False,True,441,0.0
1335,1.0,18,36.850,0,False,False,False,True,False,324,0.0
139,1.0,22,36.000,0,False,False,False,False,True,484,0.0
179,1.0,41,33.155,3,False,False,False,False,False,1681,0.0
1191,1.0,41,21.755,1,False,False,False,False,False,1681,0.0


In [10]:
pd.DataFrame(X_train, columns=X.columns).head()

,const,age,bmi,children,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest,age_sq,smoker_bmi_int
680,1.0,21,17.400,1,False,False,False,False,True,441,0.0
1335,1.0,18,36.850,0,False,False,False,True,False,324,0.0
139,1.0,22,36.000,0,False,False,False,False,True,484,0.0
179,1.0,41,33.155,3,False,False,False,False,False,1681,0.0
1191,1.0,41,21.755,1,False,False,False,False,False,1681,0.0


In [11]:
from sklearn.linear_model import Ridge
 
ridge_model = Ridge(alpha=1, fit_intercept=True).fit(X_train, y_train)
 
list(zip(X_train.columns, ridge_model.coef_))

[('const', np.float64(0.0)),
 ('age', np.float64(-15.390374428568814)),
 ('bmi', np.float64(41.83533362332057)),
 ('children', np.float64(672.7011947391875)),
 ('sex_male', np.float64(-521.671787404339)),
 ('smoker_yes', np.float64(-18023.691759159377)),
 ('region_northwest', np.float64(-78.09118392334535)),
 ('region_southeast', np.float64(-1086.5731810203865)),
 ('region_southwest', np.float64(-1248.52044496947)),
 ('age_sq', np.float64(3.422983077716937)),
 ('smoker_bmi_int', np.float64(1383.974022017071))]

In [15]:
from sklearn.preprocessing import StandardScaler
 
std = StandardScaler()
X_tr = std.fit_transform(X_train.values)
X_val = std.transform(X_valid.values)
X_te = std.transform(X_test.values)

In [14]:
from sklearn.preprocessing import StandardScaler
 
std = StandardScaler()
X_tr = std.fit_transform(X_train.values)
X_val = std.transform(X_valid.values)
X_te = std.transform(X_test.values)

In [16]:
from sklearn.linear_model import ElasticNet
 
enet_model = ElasticNet(alpha=1, l1_ratio=.5).fit(X_tr, y_train)
 
print(
    f"Train Score: {round(enet_model.score(X_tr, y_train), 4)} "
    f"Valid Score: {round(enet_model.score(X_val, y_valid), 4)} "
)
 

Train Score: 0.7628 Valid Score: 0.7758 


In [17]:
train_scores = []
val_scores = []
 
l1_ratios = np.linspace(.01, 1, 100)
 
for l1_ratio in l1_ratios:
    enet_model = ElasticNet(alpha=1, l1_ratio=l1_ratio).fit(X_tr, y_train)
    train_scores.append(enet_model.score(X_tr, y_train))
    val_scores.append(enet_model.score(X_val, y_valid))

In [19]:
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV
 
l1_ratios = np.linspace(0.01, 1, 100)
alphas = 10 ** np.linspace(-3, 3, n_alphas)
 
param_grid = {"alpha": alphas, "l1_ratio": l1_ratios}
 
eNet_model = ElasticNet(max_iter=10000, random_state=2023)
grid = GridSearchCV(eNet_model, param_grid, scoring="r2", cv=5)
 
grid.fit(X_tr, y_train)

NameError: name 'n_alphas' is not defined